## New notebook to read output of MD simulation, using Adios2 library

### part 1: **print information summary**

- Reading one of the sample output files (.bp), step by step.  
- Printing available variables and attributes, as well as their structure.

In [ ]:
import numpy as np
from adios2 import Stream
import os

def read_adios_output(outputDir, index, print_summary=True):
    attributes = {}
    variables = {}
    
    filename = os.path.join(outputDir, f"run_{index}.bp")

    with Stream(filename, "r") as s:
        for i, _ in enumerate(s.steps()):
            if i == 0:
                for attr in s.available_attributes():
                    attributes[attr] = s.read_attribute(attr)
            for var in s.available_variables():
                if var not in variables:
                    variables[var] = []
                variables[var].append(s.read(var))

    for var in variables:
        variables[var] = np.array(variables[var])

    if print_summary:
        print("Attributes Summary:")
        print("-------------------")
        for idx, (name, value) in enumerate(attributes.items(), start=1):
            print(f"{idx}. {name:<40} {value}")
        print("\nVariables Summary:")
        print("------------------")
        for idx, (name, arr) in enumerate(variables.items(), start=1):
            shape = arr.shape
            if arr.ndim == 1:
                description = shape[0]
            else:
                description = f"{shape[0]} * {list(shape[1:])}"
            print(f"{idx}. {name:<35} {description}")

    return {"attributes": attributes, "variables": variables}

output_dir = "/home/hadis/custom_vector/buildParticleOriented/buildVS/feb1_testAdios/outputs/"
result = read_adios_output(output_dir, 0)


---

### Part 2: **Read and Plot Neighbor Counts**

In [122]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
from adios2 import FileReader

def read_variable(output_dir, variable_name):

    pattern = re.compile("run_([0-9]+).bp")
    run_numbers = [int(pattern.match(x)[1]) for x in os.listdir(output_dir) if pattern.match(x)]
    
    if not run_numbers:
        raise ValueError(f"No run files found in {output_dir}")
    
    run_numbers.sort()
    variable_data = []
    temperatures = []
    run_indeces = []
    
    for i, run_num in enumerate(run_numbers):
        file_path = os.path.join(output_dir, f"run_{run_num}.bp")
        with FileReader(file_path) as reader:
            if variable_name in reader.available_variables():
                var_info = reader.available_variables()[variable_name]
                steps = int(var_info.get("AvailableStepsCount", 1))
    
                data = reader.read(variable_name, step_selection=[0, steps])
                if (data.ndim)== 1:
                    data = np.reshape(data, (1*steps, data.shape[0]//steps))
                else:
                    data = np.reshape(data, (1*steps, data.shape[0]//steps, data.shape[1]))

                variable_data.extend(data)
                
                temp_label = reader.read_attribute("temperature")
                temp_label = temp_label.flatten()
                temperatures.append(temp_label)
                
                run_index = reader.read_attribute("runIndex")
                run_index = run_index.flatten()
                run_indeces.append(run_index)
            else:
                raise ValueError(f"Variable {variable_name} not found in {file_path}")
            
    return np.array(variable_data), np.array(temperatures), np.array(run_indeces)

def plot_neighbors(neighbors_array, m_temperatures):
    neighbors_data = neighbors_array[0]
    temp_labels = neighbors_array[1]
    run_indeces = neighbors_array[2]
    
    temperature_data = m_temperatures[0]
    
    fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(7, 8))
    
    for ax in axs:
        ax.grid(True, linestyle='--', alpha=0.7, color='gray')
    
    colors = plt.cm.viridis(np.linspace(0, 1, neighbors_data.shape[2]))
    
    for i in range(neighbors_data.shape[2]):
        y = neighbors_data[:, :, i]
        axs[0].plot(y, label=f"Shell {i}", color=colors[i], linewidth=2)
    
    total_points = len(run_indeces)
    tick_indices = np.linspace(0, total_points-1, 10, dtype=int)
    
    axs[0].set_xticks(tick_indices)
    axs[0].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in tick_indices], rotation=45)
    axs[0].set_ylabel("Number of Neighbors")
    axs[0].set_title("Neighbor Analysis")
    
    axis_labels = ['x', 'y', r'$\omega$']
    
    for i, axis_label in enumerate(axis_labels):
        y = temperature_data[:, :, i]
        axs[1].plot(y, label=f"T{axis_label}", color=colors[i], linewidth=2)
    
    axs[1].set_xlabel("Temperature Labels")
    axs[1].set_xticks(tick_indices)
    axs[1].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in tick_indices], rotation=45)
    # axs[1].set_xticklabels([f"run {run_indeces[i][0]:.0f}" for i in tick_indices], rotation=45)
    axs[1].set_ylabel("Measured Temperature")

    for ax in axs:
        ax.legend(bbox_to_anchor=(1, 1), loc='upper left', frameon=True, fancybox=True, shadow=False)
    
    plt.tight_layout()
    plt.show()

def plot_energies(kinetic_energy, potential_energy, m_temperatures, show_potential=True):
    kinetic_energy_data = kinetic_energy[0]
    potential_energy_data = potential_energy[0]/ 100
    temp_labels = kinetic_energy[1]
    run_indeces = kinetic_energy[2]
    
    temperature_data = m_temperatures[0]
    
    fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(7, 8))
    
    for ax in axs:
        ax.grid(True, linestyle='--', alpha=0.7, color='gray')
    
    colors = plt.cm.plasma(np.linspace(0, 1, kinetic_energy_data.shape[2]+3))
    
    axis_labels = ['x', 'y', r'$\omega$']
    
    for i, axis_label in enumerate(axis_labels):
        y = kinetic_energy_data[:, :, i]
        axs[0].plot(y, label=f"K{axis_label}", color=colors[i])
    if show_potential:    
        y2 = potential_energy_data.flatten()
        axs[0].plot(y2, label='U', color=colors[3])
        # axs[0].plot(y2 + np.sum(kinetic_energy_data, axis=2), label='Total', color=colors[4])
    
    total_points = len(run_indeces)
    tick_indices = np.linspace(0, total_points-1, 10, dtype=int)
    
    axs[0].set_xticks(tick_indices)
    axs[0].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in tick_indices], rotation=45)
    axs[0].set_ylabel("Energy")
    axs[0].set_title("Energy Evolution")
     
    for i, axis_label in enumerate(axis_labels):
        y = temperature_data[:, :, i]
        axs[1].plot(y, label=f"T{axis_label}", color=colors[i])
    
    axs[1].set_xlabel("Temperature Labels")
    axs[1].set_xticks(tick_indices)
    axs[1].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in tick_indices], rotation=45)
    # axs[1].set_xticklabels([f"run {run_indeces[i][0]:.0f}" for i in tick_indices], rotation=45)
    axs[1].set_ylabel("Measured Temperature")

    for ax in axs:
        ax.legend(bbox_to_anchor=(1, 1), loc='upper left', frameon=True, fancybox=True, shadow=False)
    
    plt.tight_layout()
    
    plt.show()

def plot_com_velocity(com_velocity, m_temperatures):
    com_vel_data = com_velocity[0]
    temp_labels = com_velocity[1]
    run_indeces = com_velocity[2]
    
    temperature_data = m_temperatures[0]
    
    fig, axs = plt.subplots(nrows=2, ncols=1, figsize=(7, 8))
    
    for ax in axs:
        ax.grid(True, linestyle='--', alpha=0.7, color='gray')
    
    colors = plt.cm.plasma(np.linspace(0, 1, 4))
    
    axis_labels = ['x', 'y']
    # axis_labels = ['x', 'y', r'$\omega$']  #later edit COM calculation
    
    for i, axis_label in enumerate(axis_labels):
        y = com_vel_data[:, :, i]
        axs[0].plot(y, label=f"K{axis_label}", color=colors[i])
    
    total_points = len(run_indeces)
    tick_indices = np.linspace(0, total_points-1, 10, dtype=int)
    
    axs[0].set_xticks(tick_indices)
    axs[0].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in tick_indices], rotation=45)
    axs[0].set_ylabel("Velocity")
    axs[0].set_title("Center of Mass Velocity Evolution")
     
    for i, axis_label in enumerate(axis_labels):
        y = temperature_data[:, :, i]
        axs[1].plot(y, label=f"T{axis_label}", color=colors[i])
    
    axs[1].set_xlabel("Temperature Labels")
    axs[1].set_xticks(tick_indices)
    axs[1].set_xticklabels([f"T={temp_labels[i][0]:.1f}" for i in tick_indices], rotation=45)
    # axs[1].set_xticklabels([f"run {run_indeces[i][0]:.0f}" for i in tick_indices], rotation=45)
    axs[1].set_ylabel("Measured Temperature")

    for ax in axs:
        ax.legend(bbox_to_anchor=(1, 1), loc='upper left', frameon=True, fancybox=True, shadow=False)
    
    plt.tight_layout()
    
    plt.show()

def plot_trajectory(positions, t_min, t_window):
    positions = positions[0]
    t_max = t_min + t_window
    print("Size of position file = ", positions.shape)
    
    fig, ax = plt.subplots(1, 2, figsize=(8, 4), dpi=150)

    for i in range(positions.shape[1]):
        ax[0].plot(positions[t_min:t_max, i, 0], positions[t_min:t_max, i, 1], 'o', ms=1, alpha=0.6)

    ax[0].set_title("Particle Trajectories")
    ax[0].set_xlabel("X Position")
    ax[0].set_ylabel("Y Position")
    ax[0].grid(linestyle='--', alpha=0.5)

    ax[1].scatter(positions[t_min, :, 0], positions[t_min, :, 1], marker='o', color='blue', label='Initial Position')
    ax[1].scatter(positions[t_max, :, 0], positions[t_max, :, 1], marker='x', color='red', label='Final Position')

    ax[1].set_title("Initial vs. Final Positions")
    ax[1].set_xlabel("X Position")
    ax[1].set_ylabel("Y Position")
    ax[1].legend()
    ax[1].grid(linestyle='--', alpha=0.5)

    plt.tight_layout()
    plt.show()


In [118]:
number_neighbors = read_variable(output_dir, 'number of neighbors')
temperature = read_variable(output_dir, 'real temperature')

plot_neighbors(number_neighbors, temperature)


In [ ]:
kinetic = read_variable(output_dir, 'kinetic energy')
potential = read_variable(output_dir, 'potential energy')

plot_energies(kinetic, potential, temperature, show_potential=True)


In [ ]:
com_velocity = read_variable(output_dir, "center of mass velocity")

plot_com_velocity(com_velocity, temperature)

In [ ]:
positions = read_variable(output_dir, 'positions')

plot_trajectory(positions, 1188000//2, 10000)